## Ex:2 Data Wrangling and Transformation 
### Objective

To perform data wrangling and transformation on a dataset using Python and Pandas by handling missing values, removing duplicates, correcting data types, filtering data, and transforming variables into a suitable format for data analysis and machine learning.



##  Dataset Description

The dataset contains information about **student academic performance, educational background, MBA specialization, and placement salary**.

###  Dataset Attributes

| Column | Description |
|:---|:---|
| `sl_no` | Serial number / student identifier |
| `gender` | Gender of the student |
| `hsc_p` | Higher Secondary / 12th percentage |
| `hsc_s` | Higher Secondary stream |
| `degree_p` | Undergraduate degree percentage |
| `degree_t` | Undergraduate degree type |
| `etest_p` | Employability / entrance test percentage |
| `specialisation` | MBA specialization |
| `mba_p` | MBA percentage |
| `salary` | Salary offered after placement |

---

## Experiment Question

 **Using the given student placement dataset, perform data wrangling and transformation by handling missing values, scaling numerical features, detecting and treating outliers, encoding categorical variables, and generating a final model-ready dataset.**




Data wrangling and transformation is the process of converting raw student placement data into a clean, consistent, and machine-learning-ready dataset. In this experiment, the given student placement dataset is first loaded into a Pandas DataFrame and explored by examining its rows, columns, data types, and descriptive statistics. Missing values are then identified and handled by removing records with missing `salary` values and replacing missing values in `hsc_p`, `degree_p`, and `etest_p` with their respective mean values. The numerical attributes such as `hsc_p`, `degree_p`, `etest_p`, and `salary` are transformed using feature-scaling techniques such as `StandardScaler` and `MinMaxScaler` to bring the variables into suitable numerical ranges. The dataset is then divided into input features (`X`) and the target variable (`Y`), where `salary` is considered the target. The preprocessed data is saved as `Pre.csv` for further processing. Next, possible outliers in the `salary` attribute are identified using a boxplot and statistically detected using the Z-score method, where values with an absolute Z-score greater than 3 are considered potential outliers. Outliers are also treated using the capping and flooring method by calculating the 5th and 95th percentiles and replacing values outside these limits with the corresponding boundary values. The salary distribution before and after outlier treatment is then compared using visualization. Since the dataset also contains categorical attributes such as `gender`, `hsc_s`, `degree_t`, and `specialisation`, these variables are converted into numerical representations using `LabelEncoder` and One-Hot Encoding. Finally, the completely transformed dataset is verified for missing values, data types, dimensions, and numerical representation, and the resulting model-ready dataset is saved as `Final.csv`. Thus, the experiment demonstrates the complete workflow of preparing real-world tabular data for data analysis and machine-learning applications.

## Step 1: Start by importing the necessary Python libraries for data preprocessing.


In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.preprocessing import LabelEncoder
from scipy.stats import zscore
from scipy import stats
from sklearn.preprocessing import LabelEncoder

## Step 2: Load the placement dataset into a Pandas Dataframe.

In [ ]:
df = pd.read_csv("data.csv")
df.info()
df["degree_p"]

<class 'pandas.DataFrame'>
RangeIndex: 215 entries, 0 to 214
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   sl_no           215 non-null    int64  
 1   gender          215 non-null    str    
 2   hsc_p           210 non-null    float64
 3   hsc_s           215 non-null    str    
 4   degree_p        213 non-null    float64
 5   degree_t        215 non-null    str    
 6   etest_p         211 non-null    float64
 7   specialisation  215 non-null    str    
 8   mba_p           214 non-null    float64
 9   salary          148 non-null    float64
dtypes: float64(5), int64(1), str(4)
memory usage: 16.9 KB


0      58.00
1      77.48
2      64.00
3        NaN
4      73.30
       ...  
210    77.60
211    72.00
212    73.00
213    58.00
214    53.00
Name: degree_p, Length: 215, dtype: float64

In [8]:
df.head()
df.tail()

,sl_no,gender,hsc_p,hsc_s,degree_p,degree_t,etest_p,specialisation,mba_p,salary
210,211,M,82.0,Commerce,77.6,Comm&Mgmt,91.0,Mkt&Fin,74.49,400000.0
211,212,M,60.0,Science,72.0,Sci&Tech,74.0,Mkt&Fin,53.62,275000.0
212,213,M,67.0,Commerce,73.0,Comm&Mgmt,59.0,Mkt&Fin,69.72,295000.0
213,214,F,66.0,Commerce,58.0,Comm&Mgmt,70.0,Mkt&HR,60.23,204000.0
214,215,M,58.0,Science,53.0,Comm&Mgmt,89.0,Mkt&HR,60.22,NaN


In [ ]:
#df.sample(8)
df.describe()
#df.loc[0]
df.iloc[10]

sl_no                    11
gender                    M
hsc_p                   NaN
hsc_s              Commerce
degree_p               60.0
degree_t          Comm&Mgmt
etest_p                62.0
specialisation       Mkt&HR
mba_p                 60.85
salary             260000.0
Name: 10, dtype: object

In [17]:
df.shape

(215, 10)

### Brief explanation of the data inspection functions

- `pd.read_csv("Data.csv")`: Reads the CSV file and stores it as a Pandas DataFrame named `df`.
- `df.info()`: Displays the column names, data types, non-null values, and memory usage.
- `df.shape`: Returns the number of rows and columns as `(rows, columns)`.
- `df.head()`: Displays the first five rows of the DataFrame.
- `df.tail()`: Displays the last five rows of the DataFrame.
- `df.sample(8)`: Displays eight randomly selected rows from the DataFrame.
- `df.describe()`: Generates summary statistics for numerical columns, such as count, mean, standard deviation, minimum, and maximum.
- `df.loc[0]`: Selects the row with the label/index `0`.
- `df.iloc[0]`: Selects the first row by its integer position, regardless of its index label.
- `df["degree_p"]`: Selects the `degree_p` column and returns it as a Pandas Series.

## Step 3:Take a quick look at the data to understand its structure and identify any missing values or anomalies.

In [37]:
df.isnull().sum()

sl_no             0
gender            0
hsc_p             2
hsc_s             0
degree_p          1
degree_t          0
etest_p           2
specialisation    0
mba_p             0
salary            0
dtype: int64

#### The method isnull() checks each element in the DataFrame (or Series) to see if it is NaN (Not a Number) or None (missing value).
It returns a DataFrame (or Series) of the same shape as the input, with Boolean values:
#### True: The value is null (NaN or None).
#### False: The value is not null.

## Step 4: Handle Missing Data
### Option 1: If the dataset is large and only a small percentage of data is missing, you can remove rows with missing values using dropna(subset,inplace)


In [ ]:
df.dropna(subset=["salary"] , inplace = True)

In [45]:
df.isnull().sum()

sl_no             0
gender            0
hsc_p             0
hsc_s             0
degree_p          0
degree_t          0
etest_p           0
specialisation    0
mba_p             0
salary            0
dtype: int64

- `df.dropna(...)`: Removes rows that contain missing values.
- `subset=["salary"]`: Checks only the `salary` column for missing values.
- `inplace=True`: Applies the change directly to `df` instead of creating a new DataFrame.

### Option 2:If removing data isn't ideal, you can impute (df.[""].fillna(df[""].mean(),inplace)) missing values using methods like mean, median, or most frequent.

In [ ]:
df["hsc_p"]=df["hsc_p"].fillna(df["hsc_p"].mean())
df["degree_p"]=df["degree_p"].fillna(df["degree_p"].mean())
df["etest_p"]=df["etest_p"].fillna(df["etest_p"].mean()) #all the null values has been dealt with

- `df["hsc_p"]`: Selects the `hsc_p` column.
- `.mean()`: Calculates the average value of the `hsc_p` column.
- `.fillna(...)`: Replaces missing `hsc_p` values with the column average.

## Step 5: Feature Scaling
Feature scaling is the process of converting numerical features to a similar scale so that one feature does not dominate another simply because it has larger numerical values.

<img src="https://i.postimg.cc/G21gMYnF/f.png" alt="Image Description" width="500">









## Option 1( StandardScaler): This method scales the data to have a mean of 0 and a standard deviation of 1.


In [58]:
c =["hsc_p" , "degree_p" , "etest_p" , "salary"]
s1 = StandardScaler()
df[c]=s1.fit_transform(df[c])
df.head()

,sl_no,gender,hsc_p,hsc_s,degree_p,degree_t,etest_p,specialisation,mba_p,salary
0,1,M,2.265997e+00,Commerce,-1.652293,Sci&Tech,-1.328518,Mkt&HR,58.80,-0.200292
1,2,M,8.987875e-01,Science,1.346845,Sci&Tech,0.978332,Mkt&Fin,66.28,-0.951839
2,3,M,1.221419e-15,Arts,-0.728534,Comm&Mgmt,0.136149,Mkt&Fin,57.80,-0.415019
4,5,M,3.883770e-01,Commerce,0.703293,Comm&Mgmt,1.732635,Mkt&Fin,55.50,1.463849
7,8,M,-6.475513e-01,Science,-0.420614,Sci&Tech,-0.449718,Mkt&Fin,62.14,-0.393547


#### Option 2:This method scales the data to a fixed range, usually between 0 and 1. 
###  MinMaxScaler()

In [63]:
c1 =["hsc_p" , "degree_p" , "etest_p" , "salary"]
s2 = MinMaxScaler()
df[c1]=s2.fit_transform(df[c1])
df.head()

,sl_no,gender,hsc_p,hsc_s,degree_p,degree_t,etest_p,specialisation,mba_p,salary
0,1,M,0.857051,Commerce,0.057143,Sci&Tech,0.104167,Mkt&HR,58.80,0.094595
1,2,M,0.586729,Science,0.613714,Sci&Tech,0.760417,Mkt&Fin,66.28,0.000000
2,3,M,0.409023,Arts,0.228571,Comm&Mgmt,0.520833,Mkt&Fin,57.80,0.067568
4,5,M,0.485812,Commerce,0.494286,Comm&Mgmt,0.975000,Mkt&Fin,55.50,0.304054
7,8,M,0.280990,Science,0.285714,Sci&Tech,0.354167,Mkt&Fin,62.14,0.070270


## Step 6  Option 1: Identifying Outliers Using Z-Scores
The value of 3 in the context of Z-scores is often used as a threshold to identify outliers in a dataset. A Z-score represents how many standard deviations a data point is away from the mean of the dataset. Specifically:

A Z-score of 0 means the data point is exactly at the mean.
A Z-score of 1 means the data point is one standard deviation above the mean, and so on.
A Z-score of 3 corresponds to a data point being 3 standard deviations away from the mean. For a normal distribution, about 99.7% of the data points fall within 3 standard deviations of the mean (according to the 68-95-99.7 rule, which describes the spread of data in a normal distribution). Therefore, points with Z-scores greater than 3 or less than -3 are considered unusually far from the mean and are often flagged as outliers.

This threshold (Z > 3 or Z < -3) is commonly used in many statistical applications because it captures the extreme values that are rare in a normal distribution, which are typically considered to be outliers. However, the choice of threshold can vary depending on the specific application and the nature of the data.



[![Chat-GPT-Image-Aug-20-2026-08-06-05-PM.png](https://i.postimg.cc/2SFTtcXL/Chat-GPT-Image-Aug-20-2026-08-06-05-PM.png)](https://postimg.cc/4Yyz75GX)

In [80]:
columns_to_check = ["salary"]
Z_score = stats.zscore(df[columns_to_check])
u = (Z_score>3)
l = (Z_score<-3)
indx = u|l
print(l)
print(u)
clean_df = df[~indx]

[[False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 

- `columns_to_check = ["salary"]`: Specifies the numerical column to check for outliers.
- `stats.zscore(...)`: Calculates how many standard deviations each salary is from the mean.
- `u = (Z_score > 3)`: Identifies unusually high values.
- `l = (Z_score < -3)`: Identifies unusually low values.
- `indx = u | l`: Combines both conditions to create an outlier mask. `True` indicates a possible outlier.

In [81]:
clean_df.head()
clean_df.info()

<class 'pandas.DataFrame'>
Index: 145 entries, 0 to 213
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   sl_no           145 non-null    int64  
 1   gender          145 non-null    str    
 2   hsc_p           145 non-null    float64
 3   hsc_s           145 non-null    str    
 4   degree_p        145 non-null    float64
 5   degree_t        145 non-null    str    
 6   etest_p         145 non-null    float64
 7   specialisation  145 non-null    str    
 8   mba_p           145 non-null    float64
 9   salary          145 non-null    float64
dtypes: float64(5), int64(1), str(4)
memory usage: 12.5 KB


### Option 2:  Capping and Flooring Outliers
Capping and flooring is an outlier-treatment technique where extreme values are replaced with predefined boundary values instead of deleting the records.

In [92]:
l1 = df["salary"].quantile(0.05) #finds the 5th percentile used as the lower limit
u1 = df["salary"].quantile(0.95)#finds teh 95th percentile used as the upper limit
df_capped=df.copy()
df_capped["salary"]= df_capped["salary"].clip(l1,u1)
df_capped.info()

<class 'pandas.DataFrame'>
Index: 148 entries, 0 to 213
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   sl_no           148 non-null    int64  
 1   gender          148 non-null    str    
 2   hsc_p           148 non-null    float64
 3   hsc_s           148 non-null    str    
 4   degree_p        148 non-null    float64
 5   degree_t        148 non-null    str    
 6   etest_p         148 non-null    float64
 7   specialisation  148 non-null    str    
 8   mba_p           148 non-null    float64
 9   salary          148 non-null    float64
dtypes: float64(5), int64(1), str(4)
memory usage: 12.7 KB


## Step 7: Convert categorical variables into numerical format using LabelEncoder ().
[![Picture1.png](https://i.postimg.cc/yNpNvnVd/Picture1.png)](https://postimg.cc/zLW5fCHZ)




In [110]:
L1 = LabelEncoder()
df["degree_t"] = L1.fit_transform(df["degree_t"])
df.sample(10)

,sl_no,gender,hsc_p,hsc_s,degree_p,degree_t,etest_p,specialisation,mba_p,salary
59,60,M,0.314700,Science,0.460286,2,0.158333,0,56.66,0.087838
123,124,M,0.174312,Commerce,0.485714,0,0.208333,1,56.70,0.054054
147,148,M,0.494346,Commerce,0.257143,0,0.687500,0,62.28,0.135135
81,82,M,0.259654,Science,0.314286,0,0.750000,0,70.20,0.135135
200,201,M,0.195648,Commerce,0.257143,0,0.782292,0,52.81,0.135135
132,133,M,0.366332,Commerce,0.228571,0,0.508333,1,53.49,0.135135
101,102,M,0.451675,Commerce,0.342857,0,0.583333,1,60.44,0.243243
103,104,M,0.579689,Science,0.485714,2,0.729167,1,65.83,0.054054
19,20,M,0.344997,Arts,0.363771,0,0.010000,0,77.89,0.048649
11,12,M,0.374867,Commerce,0.637143,0,0.208333,0,63.70,0.067568


## Convert categorical variables into numerical format using one hot encoder

[![Picture2.png](https://i.postimg.cc/gcZ0Hv1J/Picture2.png)](https://postimg.cc/HjTHp7YD)

In [ ]:
c3 = ["gender"]
c4 = ["hsc_s"]
one_hot_encoded_data1= pd.get_dummies(df, columns = c4)
one_hot_encoded_data= pd.get_dummies(df, columns = c3)
one_hot_encoded_data1.sample(10)
one_hot_encoded_data.sample(10)

,sl_no,gender,hsc_p,degree_p,degree_t,etest_p,specialisation,mba_p,salary,hsc_s_Arts,hsc_s_Commerce,hsc_s_Science
56,57,M,0.438873,0.154286,0,0.375000,0,66.88,0.054054,False,True,False
200,201,M,0.195648,0.257143,0,0.782292,0,52.81,0.135135,False,True,False
129,130,M,0.829315,0.285714,0,0.833333,0,68.55,0.067568,False,True,False
74,75,M,0.298058,0.405714,0,0.713958,0,67.20,0.183784,False,True,False
98,99,F,0.473010,0.257143,0,0.416667,0,57.31,0.027027,False,True,False
125,126,F,0.473010,0.485714,0,0.520833,0,73.33,0.202703,False,True,False
199,200,M,0.302326,0.028571,0,0.479167,1,55.80,0.087838,False,True,False
126,127,F,0.216983,0.557143,2,0.416667,0,68.20,0.013514,False,False,True
59,60,M,0.314700,0.460286,2,0.158333,0,56.66,0.087838,False,False,True
44,45,F,0.473010,0.714286,0,0.812500,0,69.70,0.000000,False,True,False


# Exercise: Data Cleaning and Transformation – Automobile Dataset

## Step 1: Load and Explore the Dataset

### 1. Load the Dataset
- Import Pandas and load the Automobile dataset.
- Display the first 10 rows.
- Display the shape of the dataset.

### 2. Explore the Dataset
- Display the column names.
- Display the data types.
- Generate descriptive statistics.
- Identify numerical and categorical columns.
- Display unique values in categorical columns.

## Step 2: Data Cleaning

### 3. Check Missing Values
- Check for missing values in each column.
- Display the number and percentage of missing values.

### 4. Handle Missing Values
- Replace missing numerical values using mean or median.
- Replace missing categorical values using mode.
- Verify that no missing values remain.

### 5. Remove Duplicate Records
- Check for duplicate rows.
- Display the number of duplicate records.
- Remove duplicate records.
- Verify the result.

### 6. Clean the `horsepower` Column
- Identify non-numeric values such as `?`.
- Replace `?` with `NaN`.
- Convert `horsepower` to numeric.
- Handle the resulting missing values.

## Step 3: Data Transformation

### 7. Transform the `origin` Column
- Display the unique values in `origin`.
- Convert the values into meaningful labels:
  - `1` → `usa`
  - `2` → `europe`
  - `3` → `japan`

### 8. Create `weight_kg`
- Create a new column `weight_kg`.
- Convert weight from pounds to kilograms.

  `weight_kg = weight × 0.453592`

### 9. Create `mpg_category`
Create a new column based on `mpg`:
- `< 20` → `Low`
- `20–29` → `Medium`
- `≥ 30` → `High`

### 10. Create `vehicle_age`
- Create a new column `vehicle_age`.
- Assume the current year is 2026.

  `vehicle_age = 2026 - model_year`

### 11. Rename Columns
Rename:
- `mpg` → `miles_per_gallon`
- `horsepower` → `hp`
- `weight` → `weight_lbs`
- `model_year` → `year`

### 12. Filter the Data
Display vehicles:
- With `mpg > 30`
- With `horsepower > 150`
- With `cylinders >= 6`
- Manufactured after 1980
- Originating from `usa`

## Step 4: Encoding Categorical Data

### 13. Label Encoding
- Apply `LabelEncoder` to the `origin` column.
- Create a new column `origin_encoded`.
- Display the original and encoded values.
- Display the category-to-label mapping.

### 14. One-Hot Encoding
- Apply One-Hot Encoding to the `origin` column.
- Compare Label Encoding and One-Hot Encoding.
- Which encoding method is more appropriate for `origin`? Explain why.

## Step 5: Outlier Detection

### 15. Identify Outliers Using Z-Scores
- Calculate the Z-score for the numerical features.
- Identify observations with `|Z-score| > 3` as outliers.
- Count the outliers in each numerical column.
- Display the rows containing outliers.
- Decide whether the outliers should be removed or retained.

## Step 6: Feature Scaling

### 16. Standardization Using StandardScaler
- Select the numerical features.
- Apply `StandardScaler`.
- Display the standardized values.
- Verify that the features have approximately mean `0` and standard deviation `1`.

## Step 7: Normalization

### 17. Normalization Using MinMaxScaler
- Apply `MinMaxScaler` to the numerical features.
- Transform the features to the range `[0, 1]`.
- Display the normalized values.
- Compare **Standardization** and **Normalization**.
- Explain when each scaling method is appropriate.

## Step 8: Create Features and Target

### 18. Create X and Y Variables
- Select the appropriate input features as **X (independent variables)**.
- Select `mpg` as **Y (target variable)**.
- Display the shape of `X` and `Y`.
- Save `X` and `Y` into `automobile_X_Y.csv`.
- Load the CSV file again and display the first 5 rows.

## Step 9: Save the Final Dataset

### 19. Save the Preprocessed Dataset
- Combine the processed features and target variable.
- Display the final dataset.
- Check for missing values.
- Save the final dataset as `automobile_preprocessed.csv`.

In [4]:
df1 = pd.read_csv("Automobile.csv")
df1.iloc[:11]

,name,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin
0,chevrolet chevelle malibu,18.0,8.0,307.0,130.0,3504.0,12.0,70,usa
1,buick skylark 320,15.0,8.0,350.0,165.0,3693.0,11.5,70,usa
2,plymouth satellite,18.0,8.0,318.0,150.0,3436.0,11.0,70,usa
3,amc rebel sst,16.0,8.0,304.0,150.0,3433.0,12.0,70,usa
4,ford torino,17.0,NaN,302.0,140.0,3449.0,10.5,70,usa
5,ford galaxie 500,15.0,8.0,429.0,198.0,4341.0,10.0,70,usa
6,chevrolet impala,14.0,8.0,454.0,220.0,NaN,9.0,70,usa
7,plymouth fury iii,14.0,8.0,440.0,215.0,4312.0,8.5,70,usa
8,pontiac catalina,14.0,8.0,455.0,NaN,4425.0,10.0,70,usa
9,amc ambassador dpl,15.0,8.0,390.0,190.0,3850.0,8.5,70,usa


In [5]:
df1.shape

(398, 9)

In [6]:
df1.sample(10)

,name,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin
32,ford pinto,25.0,4.0,98.0,NaN,2046.0,19.0,71,usa
195,chevrolet chevette,29.0,4.0,85.0,52.0,2035.0,22.2,76,usa
194,amc hornet,22.5,6.0,232.0,90.0,3085.0,17.6,76,usa
220,datsun f-10 hatchback,33.5,4.0,85.0,70.0,1945.0,16.8,77,japan
211,mercedes-benz 280s,16.5,6.0,168.0,120.0,3820.0,16.7,76,europe
279,honda accord lx,29.5,4.0,98.0,68.0,2135.0,16.6,78,japan
71,mazda rx2 coupe,19.0,3.0,70.0,97.0,2330.0,13.5,72,japan
178,peugeot 504,23.0,4.0,120.0,88.0,2957.0,17.0,75,europe
343,toyota starlet,39.1,4.0,79.0,58.0,1755.0,16.9,81,japan
55,volkswagen model 111,27.0,4.0,97.0,60.0,1834.0,19.0,71,europe


In [7]:
df1.info()

<class 'pandas.DataFrame'>
RangeIndex: 398 entries, 0 to 397
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   name          398 non-null    str    
 1   mpg           398 non-null    float64
 2   cylinders     395 non-null    float64
 3   displacement  395 non-null    float64
 4   horsepower    386 non-null    float64
 5   weight        396 non-null    float64
 6   acceleration  395 non-null    float64
 7   model_year    398 non-null    int64  
 8   origin        398 non-null    str    
dtypes: float64(6), int64(1), str(2)
memory usage: 28.1 KB


In [9]:
df1.describe()

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year
count,398.000000,395.000000,395.000000,386.000000,396.000000,395.000000,398.000000
mean,23.514573,5.445570,193.340506,104.316062,2965.025253,15.562278,76.010050
std,7.815984,1.696203,104.425993,38.086281,845.254458,2.750260,3.697627
min,9.000000,3.000000,68.000000,46.000000,1613.000000,8.000000,70.000000
25%,17.500000,4.000000,102.500000,75.250000,2222.250000,13.850000,73.000000
50%,23.000000,4.000000,146.000000,92.500000,2797.500000,15.500000,76.000000
75%,29.000000,8.000000,262.000000,125.000000,3581.750000,17.150000,79.000000
max,46.600000,8.000000,455.000000,230.000000,5140.000000,24.800000,82.000000


In [10]:
numerical_columns = df1.select_dtypes(include="number").columns
categorical_columns = df1.select_dtypes(exclude="number").columns

print("Numerical columns:", list(numerical_columns))
print("Categorical columns:", list(categorical_columns))

Numerical columns: ['mpg', 'cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model_year']
Categorical columns: ['name', 'origin']


In [11]:
for column in categorical_columns:
    print(df1[column].unique())

<StringArray>
[ 'chevrolet chevelle malibu',          'buick skylark 320',
         'plymouth satellite',              'amc rebel sst',
                'ford torino',           'ford galaxie 500',
           'chevrolet impala',          'plymouth fury iii',
           'pontiac catalina',         'amc ambassador dpl',
 ...
 'chrysler lebaron medallion',             'ford granada l',
           'toyota celica gt',          'dodge charger 2.2',
           'chevrolet camaro',            'ford mustang gl',
                  'vw pickup',              'dodge rampage',
                'ford ranger',                 'chevy s-10']
Length: 305, dtype: str
<StringArray>
['usa', 'japan', 'europe']
Length: 3, dtype: str


In [12]:
df1.isnull().sum()

name             0
mpg              0
cylinders        3
displacement     3
horsepower      12
weight           2
acceleration     3
model_year       0
origin           0
dtype: int64

In [13]:
df1["horsepower"] = df1["horsepower"].fillna(df1["horsepower"].mean())
df1["displacement"] = df1["displacement"].fillna(df1["displacement"].median())
df1["weight"] = df1["weight"].fillna(df1["weight"].mean())
df1["acceleration"] = df1["acceleration"].fillna(df1["acceleration"].median())
df1["cylinders"] = df1["cylinders"].fillna(df1["cylinders"].mean())

In [14]:
print(df1.duplicated().sum())
df1 = df1.drop_duplicates()

0


In [15]:
df1["horsepower"] = df1["horsepower"].replace("?" , 100)
df1[df1["horsepower"] == "?"]

,name,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin


In [16]:
df1["origin"] = df1["origin"].replace({
    "usa": 1,
    "europe":2,
    "japan":3
})

df1.sample(5)

,name,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin
239,subaru dl,30.0,4.00000,97.0,67.000000,1985.0,16.4,77,3
335,triumph tr7 coupe,35.0,4.00000,122.0,88.000000,2500.0,15.1,80,2
8,pontiac catalina,14.0,8.00000,455.0,104.316062,4425.0,10.0,70,1
368,chevrolet cavalier wagon,27.0,4.00000,112.0,88.000000,2640.0,18.6,82,1
269,dodge omni,30.9,5.44557,105.0,75.000000,2230.0,14.5,78,1


In [17]:
df1["origin"] = df1["origin"].replace({
    1: "usa",
    2:"europe",
    3:"japan"
})
df1.sample(5)

,name,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin
147,fiat 128,24.0,4.0,90.0,75.0,2108.0,15.5,74,europe
181,honda civic cvcc,33.0,4.0,91.0,53.0,1795.0,17.5,75,japan
315,amc concord,24.3,4.0,151.0,90.0,3003.0,20.1,80,usa
221,chevrolet caprice classic,17.5,8.0,305.0,145.0,3880.0,12.5,77,usa
363,buick century,22.4,6.0,231.0,110.0,3415.0,15.8,81,usa


In [18]:
df1["Weight_kg"] = df1["weight"]*0.453592
df1.sample(5)

,name,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,Weight_kg
149,honda civic,24.0,4.0,120.0,97.0,2489.0,15.0,74,japan,1128.990488
382,toyota corolla,34.0,4.0,108.0,70.0,2245.0,16.9,82,japan,1018.314040
334,mazda rx-7 gs,23.7,3.0,70.0,100.0,2420.0,12.5,80,japan,1097.692640
228,ford granada,18.5,6.0,250.0,98.0,3525.0,19.0,77,usa,1598.911800
48,ford mustang,18.0,6.0,250.0,88.0,3139.0,14.5,71,usa,1423.825288


In [19]:
df1["mpg_category"] = np.where(
    df1["mpg"] < 20 , "Low" , 
    np.where(df1["mpg"] < 30 , "Medium" , "High")
)
df1.sample(5)

,name,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,Weight_kg,mpg_category
9,amc ambassador dpl,15.0,8.0,390.0,190.0,3850.0,8.5,70,usa,1746.329200,Low
134,amc matador,16.0,6.0,258.0,110.0,3632.0,18.0,74,usa,1647.446144,Low
175,volkswagen rabbit,29.0,4.0,90.0,70.0,1937.0,14.0,75,europe,878.607704,Medium
151,fiat x1.9,31.0,4.0,79.0,67.0,2000.0,16.0,74,europe,907.184000,High
16,amc hornet,18.0,6.0,199.0,97.0,2774.0,15.5,70,usa,1258.264208,Low


In [20]:
df1 = df1.rename(columns = {"year":"model_year"})
df1["model_year"] = np.where(
    df1["model_year"]<100 , 1900+df1["model_year"] , df1["model_year"]
)
df1["Vehicle_age"] = 2026 - df1["model_year"]
df1.sample(5)

,name,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,Weight_kg,mpg_category,Vehicle_age
383,honda civic,38.0,4.0,91.0,67.000000,1965.0,15.0,1982,japan,891.308280,High,44
265,dodge magnum xe,17.5,8.0,146.0,140.000000,4080.0,13.7,1978,usa,1850.655360,Low,48
249,oldsmobile cutlass salon brougham,19.9,8.0,260.0,110.000000,3365.0,15.5,1978,usa,1526.337080,Low,48
136,ford gran torino,16.0,8.0,302.0,104.316062,4141.0,14.0,1974,usa,1878.324472,Low,52
5,ford galaxie 500,15.0,8.0,429.0,198.000000,4341.0,10.0,1970,usa,1969.042872,Low,56


In [21]:
df1 = df1.rename(columns = {"mpg":"miles_per_gallon" , "horsepower":"hp" , "weight":"weight_lbs" , "model_year":"year"})
df1.info()

<class 'pandas.DataFrame'>
RangeIndex: 398 entries, 0 to 397
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   name              398 non-null    str    
 1   miles_per_gallon  398 non-null    float64
 2   cylinders         398 non-null    float64
 3   displacement      398 non-null    float64
 4   hp                398 non-null    float64
 5   weight_lbs        398 non-null    float64
 6   acceleration      398 non-null    float64
 7   year              398 non-null    int64  
 8   origin            398 non-null    object 
 9   Weight_kg         398 non-null    float64
 10  mpg_category      398 non-null    str    
 11  Vehicle_age       398 non-null    int64  
dtypes: float64(7), int64(2), object(1), str(2)
memory usage: 37.4+ KB


In [22]:
filtered_df = df1[
    (df1["miles_per_gallon"] > 30) &
    (df1["hp"] > 150) &
    (df1["cylinders"] >= 6) &
    (df1["year"] > 1980) &
    (df1["origin"] == "usa")
]

filtered_df

,name,miles_per_gallon,cylinders,displacement,hp,weight_lbs,acceleration,year,origin,Weight_kg,mpg_category,Vehicle_age


In [25]:
l2 = LabelEncoder()
df1["Origin_Encoded"] = l2.fit_transform(df1["origin"])
df1.sample(5)

,name,miles_per_gallon,cylinders,displacement,hp,weight_lbs,acceleration,year,origin,Weight_kg,mpg_category,Vehicle_age,Origin_Encoded
39,pontiac catalina brougham,14.0,8.0,400.0,175.0,4464.0,11.5,1971,usa,2024.834688,Low,55,2
207,volvo 245,20.0,4.0,130.0,102.0,3150.0,15.7,1976,europe,1428.814800,Medium,50,0
300,oldsmobile cutlass salon brougham,23.9,8.0,260.0,90.0,3420.0,22.2,1979,usa,1551.284640,Medium,47,2
60,chevrolet vega,20.0,4.0,140.0,90.0,2408.0,19.5,1972,usa,1092.249536,Medium,54,2
196,chevrolet woody,24.5,4.0,98.0,60.0,2164.0,22.1,1976,usa,981.573088,Medium,50,2


In [32]:
C1 = ["origin"]
One_Hot_Encoded_Data1 = pd.get_dummies(df1 , columns = C1)
One_Hot_Encoded_Data1.sample(5)

,name,miles_per_gallon,cylinders,displacement,hp,weight_lbs,acceleration,year,Weight_kg,mpg_category,Vehicle_age,Origin_Encoded,origin_europe,origin_japan,origin_usa
132,chevrolet vega,25.0,4.0,140.0,75.0,2542.0,17.0,1974,1153.030864,Medium,52,2,False,False,True
282,ford fairmont 4,22.3,4.0,146.0,88.0,2890.0,17.3,1979,1310.880880,Medium,47,2,False,False,True
392,chevrolet camaro,27.0,4.0,151.0,90.0,2950.0,17.3,1982,1338.096400,Medium,44,2,False,False,True
382,toyota corolla,34.0,4.0,108.0,70.0,2245.0,16.9,1982,1018.314040,High,44,1,False,True,False
150,subaru,26.0,4.0,108.0,93.0,2391.0,15.5,1974,1084.538472,Medium,52,1,False,True,False


In [52]:
z_scores_hp = stats.zscore(df1["hp"])
z_scores_miles_per_gallon = stats.zscore(df1["miles_per_gallon"])
z_scores_cylinders = stats.zscore(df1["cylinders"])
z_scores_displacement = stats.zscore(df1["displacement"])
z_scores_weight = stats.zscore(df1["weight_lbs"])
z_scores_acceleration = stats.zscore(df1["acceleration"])

outlier_mask_hp = np.abs(z_scores_hp) > 3
outlier_mask_mpg = np.abs(z_scores_miles_per_gallon) > 3
outlier_mask_cylinders = np.abs(z_scores_cylinders) > 3
outlier_mask_displacement = np.abs(z_scores_displacement) > 3
outlier_mask_weight = np.abs(z_scores_weight) > 3
outlier_mask_acceleration = np.abs(z_scores_acceleration) > 3

print("Number of hp outliers:", outlier_mask_hp.sum())
print("Number of outliers:", outlier_mask_mpg.sum())
print("Number of outliers:", outlier_mask_cylinders.sum())
print("Number of outliers:", outlier_mask_displacement.sum())
print("Number of outliers:", outlier_mask_weight.sum())
print("Number of outliers:", outlier_mask_acceleration.sum())

hp_outliers = df1[outlier_mask_hp]
acceleration_outliers = df1[outlier_mask_acceleration]
hp_outliers
acceleration_outliers


Number of hp outliers: 4
Number of outliers: 0
Number of outliers: 0
Number of outliers: 0
Number of outliers: 0
Number of outliers: 2


,name,miles_per_gallon,cylinders,displacement,hp,weight_lbs,acceleration,year,origin,Weight_kg,mpg_category,Vehicle_age,Origin_Encoded
299,peugeot 504,27.2,4.0,141.0,71.0,3190.0,24.8,1979,europe,1446.95848,Medium,47,0
394,vw pickup,44.0,4.0,97.0,52.0,2130.0,24.6,1982,europe,966.15096,High,44,0


In [53]:
C5 = ["hp" , "miles_per_gallon" , "cylinders" , "displacement" , "weight_lbs" , "acceleration"]
S2 = StandardScaler()
df1[C5] = S2.fit_transform(df1[C5])
df1.sample(5)

,name,miles_per_gallon,cylinders,displacement,hp,weight_lbs,acceleration,year,origin,Weight_kg,mpg_category,Vehicle_age,Origin_Encoded
294,maxda glc deluxe,1.356035,-0.856554,-1.028881,-1.049573,-1.175714,-0.132220,1979,japan,895.844200,High,47,1
117,fiat 128,0.702705,-0.856554,-1.201991,-1.476705,-1.303971,1.439182,1973,europe,846.856264,Medium,53,0
244,volkswagen rabbit custom diesel,2.508971,-0.856554,-0.990413,-1.503401,-1.163839,2.170066,1978,europe,900.380120,High,48,0
280,pontiac lemans v6,-0.258075,0.328521,0.365610,0.285216,0.332487,-0.059132,1979,usa,1471.906040,Medium,47,2
141,audi fox,0.702705,-0.856554,-0.913475,-0.569049,-0.885950,0.342855,1974,europe,1006.520648,Medium,52,0


In [62]:
C6 = ["hp" , "miles_per_gallon" , "cylinders" , "displacement" , "weight_lbs" , "acceleration"]
S3 = MinMaxScaler()
df1[C6] = S3.fit_transform(df1[C6])
df1.sample(5)

,name,miles_per_gallon,cylinders,displacement,hp,weight_lbs,acceleration,year,origin,Weight_kg,mpg_category,Vehicle_age,Origin_Encoded
177,audi 100ls,0.372340,0.2,0.121447,0.266304,0.306493,0.416667,1975,europe,1221.976848,Medium,51,0
74,ford gran torino (sw),0.106383,1.0,0.604651,0.510870,0.760136,0.476190,1972,usa,1947.724048,Low,54,2
83,dodge colt (sw),0.505319,0.2,0.077519,0.184783,0.156223,0.416667,1972,usa,981.573088,Medium,54,2
167,toyota corolla,0.531915,0.2,0.074935,0.157609,0.158208,0.476190,1975,japan,984.748232,Medium,51,1
134,amc matador,0.186170,0.6,0.490956,0.347826,0.572441,0.595238,1974,usa,1647.446144,Low,52,2


In [65]:
X = One_Hot_Encoded_Data1.drop(columns=["miles_per_gallon", "name", "Origin_Encoded"])

Y = One_Hot_Encoded_Data1["miles_per_gallon"]

print("X shape:", X.shape)
print("Y shape:", Y.shape)

automobile_X_Y = pd.concat([X, Y], axis=1)
automobile_X_Y.to_csv("automobile_X_Y.csv", index=False)


loaded_data = pd.read_csv("automobile_X_Y.csv")
loaded_data.head()

X shape: (398, 12)
Y shape: (398,)


,cylinders,displacement,hp,weight_lbs,acceleration,year,Weight_kg,mpg_category,Vehicle_age,origin_europe,origin_japan,origin_usa,miles_per_gallon
0,8.00000,307.0,130.0,3504.0,12.0,1970,1589.386368,Low,56,False,False,True,18.0
1,8.00000,350.0,165.0,3693.0,11.5,1970,1675.115256,Low,56,False,False,True,15.0
2,8.00000,318.0,150.0,3436.0,11.0,1970,1558.542112,Low,56,False,False,True,18.0
3,8.00000,304.0,150.0,3433.0,12.0,1970,1557.181336,Low,56,False,False,True,16.0
4,5.44557,302.0,140.0,3449.0,10.5,1970,1564.438808,Low,56,False,False,True,17.0


In [67]:
final_dataset = automobile_X_Y.copy()
final_dataset.head()
print(final_dataset.isnull().sum())
final_dataset.to_csv("automobile_preprocessed.csv", index=False)

cylinders           0
displacement        0
hp                  0
weight_lbs          0
acceleration        0
year                0
Weight_kg           0
mpg_category        0
Vehicle_age         0
origin_europe       0
origin_japan        0
origin_usa          0
miles_per_gallon    0
dtype: int64
